# 01 Loading and Smoothing

This notebook is a short tutorial for the first two building blocks of the particle-prediction package:

- loading latent trajectories into `EmbryoTrajectory` objects
- smoothing those trajectories with a time-scaled Savitzky-Golay filter

The notebook is organized in two parts:

1. a synthetic example that runs anywhere and shows the API end to end
2. an optional real-data section that uses the current loader if you point it at build output files

At this stage, the notebook focuses on inspection of arrays, metadata, and smoothing diagnostics rather than visualization modules.

## What You Should Look For

As you work through the notebook, check three things:

1. raw trajectories preserve experiment metadata and original timestamps
2. `window_seconds` is converted into a sensible odd frame window using `delta_t`
3. smoothing reduces small frame-to-frame noise without mutating the raw trajectory object

The synthetic walkthrough is the easiest place to verify behavior before trying real data.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

project_root = Path.cwd()
while project_root != project_root.parent and not (project_root / "dev" / "particle_prediction" / "data" / "loading.py").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dev.particle_prediction.data.loading import EmbryoTrajectory, load_trajectories
from dev.particle_prediction.data.smoothing import smooth_trajectory, smooth_trajectories

pd.set_option("display.max_columns", 20)
pd.set_option("display.precision", 4)

print(f"Project root: {project_root}")

Project root: /net/trapnell/vol1/home/nlammers/projects/repositories/morphseq


## Part 1: Synthetic Walkthrough

This section builds a small noisy trajectory directly in PCA space. Use it to confirm that the smoothing API behaves the way the docs describe, even before loading any real CSV files.

In [2]:
time_seconds = np.arange(9, dtype=np.float64) * 60.0
true_signal = np.column_stack(
    [
        0.05 * (np.arange(9, dtype=np.float64) - 4.0) ** 2,
        0.30 * np.arange(9, dtype=np.float64) + 1.0,
    ]
)
noise = np.array(
    [
        [0.00, 0.00],
        [0.10, -0.08],
        [-0.12, 0.07],
        [0.08, -0.05],
        [0.00, 0.06],
        [-0.07, -0.04],
        [0.09, 0.05],
        [-0.10, -0.06],
        [0.00, 0.00],
    ],
    dtype=np.float64,
)
noisy_signal = true_signal + noise

synthetic_trajectory = EmbryoTrajectory(
    embryo_id="synthetic_embryo",
    trajectory=noisy_signal,
    time_seconds=time_seconds,
    delta_t=60.0,
    temperature=28.5,
    perturbation_class="wildtype",
    experiment_id="synthetic_exp",
)

synthetic_smoothed = smooth_trajectory(
    synthetic_trajectory,
    window_seconds=5.0 * 60.0,
    poly_order=2,
)

comparison_df = pd.DataFrame(
    {
        "time_seconds": time_seconds,
        "raw_dim0": synthetic_trajectory.trajectory[:, 0],
        "smoothed_dim0": synthetic_smoothed.smoothed[:, 0],
        "raw_dim1": synthetic_trajectory.trajectory[:, 1],
        "smoothed_dim1": synthetic_smoothed.smoothed[:, 1],
    }
)

print("Smoothing diagnostics:")
print(synthetic_smoothed.diagnostics)
print()
print("First few rows of raw vs smoothed values:")
comparison_df.head()

Smoothing diagnostics:
{'delta_t': 60.0, 'trajectory_length': 9, 'window_seconds': 300.0, 'window_frames': 5, 'effective_poly_order': 2}

First few rows of raw vs smoothed values:


,time_seconds,raw_dim0,smoothed_dim0,raw_dim1,smoothed_dim1
0,0.0,0.80,0.8246,1.00,0.9857
1,60.0,0.55,0.4597,1.22,1.2771
2,120.0,0.08,0.2034,1.67,1.5843
3,180.0,0.13,0.0451,1.85,1.9306
4,240.0,0.00,0.0060,2.26,2.1880


## Part 2: Real-Data Run for `20251207_pbx`

This section is now preconfigured for one concrete experiment so you can inspect the real loader and smoother outputs immediately.

What to check here:

- how many trajectories are loaded for this experiment
- whether `delta_t` is consistent across embryos
- how `window_seconds` maps to `window_frames` for the selected subset
- whether residual magnitudes look reasonable after smoothing

In [5]:
build_dir = Path("/net/trapnell/vol1/home/mdcolon/proj/morphseq/morphseq_playground/metadata/build06_output")
experiment_ids = ["20251207_pbx"]
window_seconds = 10.0 * 60.0
poly_order = 2

real_dataset = None
real_smoothed = None

if build_dir.exists():
    real_dataset = load_trajectories(
        build_dir=build_dir,
        experiment_ids=experiment_ids,
        n_components=10,
        min_trajectory_length=3,
        verbose=False,
    )
    subset = real_dataset.trajectories[:5]
    real_smoothed = smooth_trajectories(
        subset,
        window_seconds=window_seconds,
        poly_order=poly_order,
    )

    dataset_summary = pd.DataFrame(
        {
            "embryo_id": [traj.embryo_id for traj in subset],
            "experiment_id": [traj.experiment_id for traj in subset],
            "n_timepoints": [len(traj.time_seconds) for traj in subset],
            "delta_t": [traj.delta_t for traj in subset],
            "class": [traj.perturbation_class for traj in subset],
            "window_frames": [traj_s.window_frames for traj_s in real_smoothed],
        }
    )

    print(f"Build dir: {build_dir}")
    print(f"Experiment IDs: {experiment_ids}")
    print(f"Loaded {len(real_dataset)} trajectories")
    dataset_summary
else:
    print("Configured build_dir does not exist.")
    print(f"Current value: {build_dir}")

Build dir: /net/trapnell/vol1/home/mdcolon/proj/morphseq/morphseq_playground/metadata/build06_output
Experiment IDs: ['20251207_pbx']
Loaded 94 trajectories


## Part 3: Inspect the Effect of Smoothing

Use the final cell to compare raw and smoothed trajectories numerically. The main quantities to check are:

- the chosen frame window for each trajectory
- the residual scale after smoothing
- whether the raw source object is unchanged
- whether the smoothed output has the same length and time axis as the source

In [6]:
synthetic_error_before = np.mean((synthetic_trajectory.trajectory - true_signal) ** 2)
synthetic_error_after = np.mean((synthetic_smoothed.smoothed - true_signal) ** 2)

print("Synthetic example summary")
print(f"  window_frames: {synthetic_smoothed.window_frames}")
print(f"  MSE before smoothing: {synthetic_error_before:.6f}")
print(f"  MSE after smoothing:  {synthetic_error_after:.6f}")
print(f"  source unchanged: {np.array_equal(synthetic_trajectory.trajectory, noisy_signal)}")

if real_smoothed is not None:
    real_summary = pd.DataFrame(
        {
            "embryo_id": [traj.source.embryo_id for traj in real_smoothed],
            "window_frames": [traj.window_frames for traj in real_smoothed],
            "mean_abs_residual": [float(np.mean(np.abs(traj.residuals))) for traj in real_smoothed],
            "n_timepoints": [len(traj.time_seconds) for traj in real_smoothed],
        }
    )
    print()
    print("Real-data smoothing summary")
    real_summary
else:
    print()
    print("Real-data example not run yet. Update `build_dir` and rerun Part 2 when ready.")

Synthetic example summary
  window_frames: 5
  MSE before smoothing: 0.004383
  MSE after smoothing:  0.000301
  source unchanged: True

Real-data smoothing summary
